# RevalExo acceleration-reference correction

The internal model uses gravity-inclusive acceleration magnitude. Rebuild the external windows from RevalExo `acceleration` rather than gravity-removed `free_acceleration`, while preserving the verified mapping and units.

In [1]:
from pathlib import Path
import ast, numpy as np, pandas as pd, h5py
ROOT=Path('../data/raw/revalexo').resolve(); OUT=Path('../data/processed'); FS=100.; WIN=500; HOP=250; G=9.80665; RAD2DEG=180/np.pi
ORDER=['Pelvis','Right Upper Leg','Right Lower Leg','Right Foot','Left Upper Leg','Left Lower Leg','Left Foot']; windows=[]; rows=[]
for folder in sorted(ROOT.glob('raw_full_part*/raw_full/Subject*')):
 group=folder.name.rsplit('_',1)[-1]
 if group not in {'HC','ST'}: continue
 ann=pd.read_csv(folder/'annotations.csv'); ann=ann[ann.Task.str.contains('level ground walking',case=False,na=False)]
 with h5py.File(folder/'mvn-analyze.hdf5','r') as f:
  g=f['mvn-analyze/xsens-motion-trackers']; acc=g['acceleration']; gyro=g['gyroscope']; heads=ast.literal_eval(''.join(list(acc.attrs['Data headings']))); assert heads==ORDER
  t=g['process_time_s'][:].ravel(); source_hz=1/np.median(np.diff(t)); a=np.asarray(acc[:,[0,3,6],:],dtype='float32')/G; q=np.asarray(gyro[:,[0,3,6],:],dtype='float32')*RAD2DEG
  sig=np.concatenate([a[:,0],q[:,0],a[:,2],q[:,2],a[:,1],q[:,1]],axis=1); target=np.arange(t[0],t[-1],1/FS); res=np.column_stack([np.interp(target,t,sig[:,j]) for j in range(18)]).astype('float32')
 for _,w in ann.iterrows():
  lo=max(0,int(np.searchsorted(target,float(w.Start_toa_s)))); hi=min(len(target),int(np.searchsorted(target,float(w.End_toa_s))))
  for s in range(lo,hi-WIN+1,HOP):
   wid=len(windows); windows.append(res[s:s+WIN]); rows.append({'window_id':wid,'subject':folder.name,'group':group,'window_start_index':s,'window_seconds':5.0,'source_hz':source_hz})
arr=np.asarray(windows,dtype='float32'); meta=pd.DataFrame(rows); assert arr.shape[1:]==(500,18) and np.isfinite(arr).all(); np.save(OUT/'revalexo_external_windows_float32.npy',arr); meta.to_csv(OUT/'revalexo_external_window_metadata.csv',index=False); print(arr.shape); print(meta.groupby('group').size())

(2228, 500, 18)
group
HC     754
ST    1474
dtype: int64
